# Compare Python output with AcqKnowledge

Read the reference `.acq` directly with `bioread` and compare explicitly selected channels. No custom functions, resampling, alignment or automatic unit conversion. Use matching recordings and segments: sample zero must represent the same instant.

Dependencies: `bioread numpy pandas h5py`. No results are included because no reference recording was supplied.

### Imports

In [ ]:
from pathlib import Path
import hashlib
import json
import bioread
import h5py
import numpy as np
import pandas as pd

### Input and report files

In [ ]:
python_file = Path("output/P001/P001_1_video_filter_edit.h5")
reference_file = Path("reference/P001_1_video_filter_edit.acq")
report_file = Path("comparison.json")

### Read the Python output

In [ ]:
python_signals, python_info = {}, {}
with h5py.File(python_file, "r") as saved:
    assert saved.attrs["format"] == "acq-emg-pipeline-v1"
    metadata = json.loads(saved["metadata_json"][()])
    for name, dataset in saved["signals"].items():
        python_signals[name] = dataset[:]
        python_info[name] = dict(dataset.attrs)
pd.DataFrame(python_info).T

### Read the reference ACQ

In [ ]:
reference = bioread.read_file(str(reference_file))

### List reference channels

In [ ]:
pd.DataFrame([{"index": i, "order_num": channel.order_num, "name": channel.name,
               "units": channel.units, "rate_hz": channel.samples_per_second,
               "samples": len(channel.data)} for i, channel in enumerate(reference.channels)])

### Match the channels

Enter reference **indices** from the table. Intermediate keys such as `ZM_bandpass` can also be compared if the ACQ retains those stages.

In [ ]:
reference_indices = {"ZM_rms": None, "CS_rms": None,
                     "ZM_pct_mvc": None, "CS_pct_mvc": None}

### Select reference samples

In [ ]:
assert reference_indices and len(set(reference_indices.values())) == len(reference_indices)
reference_signals, reference_channels = {}, {}
for name, index in reference_indices.items():
    assert name in python_signals and type(index) is int and 0 <= index < len(reference.channels)
    reference_channels[name] = reference.channels[index]
    reference_signals[name] = np.asarray(reference.channels[index].data, dtype=float)

### Optional explicit unit conversion

For a reference in V and Python in mV, set that channel’s scale to `1000` and expected reference units to `"V"`. Otherwise keep 1.

In [ ]:
reference_scales = {name: 1.0 for name in reference_indices}
expected_reference_units = {name: python_info[name]["units"] for name in reference_indices}

### Apply the declared scales

In [ ]:
scaled_reference = {}
for name, values in reference_signals.items():
    assert np.isfinite(reference_scales[name]) and reference_scales[name] > 0
    scaled_reference[name] = values * reference_scales[name]

### Check lengths, sampling rates and units

In [ ]:
structure_rows = []
for name, channel in reference_channels.items():
    actual, expected = python_signals[name], scaled_reference[name]
    structure_rows.append({"signal": name, "same_length": len(actual) == len(expected),
        "same_rate": bool(np.isclose(python_info[name]["sample_rate_hz"], channel.samples_per_second, rtol=1e-12, atol=0)),
        "expected_units": channel.units == expected_reference_units[name],
        "finite_samples": bool(np.isfinite(actual).all() and np.isfinite(expected).all()),
        "nonempty": len(actual) > 0 and len(expected) > 0})
structure = pd.DataFrame(structure_rows).set_index("signal")
structure

### Stop if signals are incompatible

In [ ]:
assert structure.to_numpy().all(), "Fix the mismatches above; signals will not be truncated or resampled"

### Set comparison tolerances

Examples, not validated acceptance limits. Absolute tolerances use each Python channel’s units.

In [ ]:
absolute_tolerance = {name: 1e-8 for name in reference_indices}
relative_tolerance = 1e-5

### Calculate sample differences

In [ ]:
differences = {name: python_signals[name] - scaled_reference[name] for name in reference_indices}

### Calculate sample tolerances

In [ ]:
assert np.isfinite(relative_tolerance) and relative_tolerance >= 0
tolerances = {}
for name in reference_indices:
    assert np.isfinite(absolute_tolerance[name]) and absolute_tolerance[name] >= 0
    tolerances[name] = absolute_tolerance[name] + relative_tolerance * np.abs(scaled_reference[name])

### Summarize the errors

In [ ]:
error_rows = []
for name, difference in differences.items():
    worst = int(np.argmax(np.abs(difference)))
    error_rows.append({"signal": name, "units": python_info[name]["units"],
        "max_absolute_error": float(np.max(np.abs(difference))),
        "rmse": float(np.sqrt(np.mean(difference ** 2))), "bias": float(np.mean(difference)),
        "worst_sample": worst, "worst_time_s": worst / python_info[name]["sample_rate_hz"],
        "samples_outside_tolerance": int(np.sum(np.abs(difference) > tolerances[name])),
        "pass": bool(np.all(np.abs(difference) <= tolerances[name]))})
results = pd.DataFrame(error_rows).set_index("signal")
results

### Overall signal result

PASS applies only to the mapped signals. Event markers and journal text are separate from this result.

In [ ]:
signals_match = bool(results["pass"].all())
print("PASS" if signals_match else "FAIL")

### List Python event times

In [ ]:
python_events = pd.DataFrame(metadata.get("markers", []))
python_events

### List reference event times

In [ ]:
reference_events = pd.DataFrame([{"text": event.text, "time_s": float(event.time_index)}
                                 for event in (reference.event_markers or [])], columns=["text", "time_s"])
reference_events

### Compare event times by label and occurrence

Diagnostic only; types and channel assignments are not matched.

In [ ]:
python_events = python_events.reindex(columns=["text", "time_s"]).sort_values("time_s")
reference_events = reference_events.sort_values("time_s")
python_events["occurrence"] = python_events.groupby("text").cumcount()
reference_events["occurrence"] = reference_events.groupby("text").cumcount()
event_comparison = python_events.merge(reference_events, on=["text", "occurrence"], how="outer",
                                       suffixes=("_python", "_reference"), indicator=True)
event_comparison["difference_s"] = event_comparison["time_s_python"] - event_comparison["time_s_reference"]
event_comparison

### Record the compared file hashes

In [ ]:
with python_file.open("rb") as source:
    python_hash = hashlib.file_digest(source, "sha256").hexdigest()
with reference_file.open("rb") as source:
    reference_hash = hashlib.file_digest(source, "sha256").hexdigest()

### Build the report

In [ ]:
report = {"signal_pass": signals_match, "python_file": str(python_file.resolve()),
    "reference_file": str(reference_file.resolve()), "python_sha256": python_hash,
    "reference_sha256": reference_hash, "reference_indices": reference_indices,
    "reference_scales": reference_scales, "absolute_tolerance": absolute_tolerance,
    "relative_tolerance": relative_tolerance, "processing": metadata.get("processing"),
    "results": error_rows, "marker_diagnostics": json.loads(event_comparison.to_json(orient="records")),
    "scope": "Mapped signals, sample zero to sample zero; markers do not affect PASS"}

### Save the report

In [ ]:
with report_file.open("x") as saved:
    json.dump(report, saved, indent=2, allow_nan=False)
report_file.resolve()

[bioread](https://github.com/uwmadison-chm/bioread) · [BIOPAC guide](https://www.biopac.com/wp-content/uploads/AcqKnowledge-5-Software-Guide.pdf)